# Triage runbook

The same steps as the scripts, for running and inspecting in VS Code. All logic lives in `src/triage/`; this notebook only calls it, so the notebook and the production jobs always behave the same.

Kernel: this project's `.venv` (`uv sync` then `uv pip install -e .`).

In [ ]:
import pandas as pd
from triage.config import load_config, resolve
from triage.clean import load_raw, clean_rows, clean_text
from triage import registry

cfg = load_config('../config.yaml')
models_dir = resolve(cfg, cfg['paths']['models_dir'])

## 1. Clean the export
Check what was removed and what the cleaned text looks like.

In [ ]:
raw = load_raw(cfg)
df, removed = clean_rows(raw, cfg)
print(f'{len(raw):,} rows -> {len(df):,} clean tickets'); removed

In [ ]:
df[[cfg['data']['short_description_col'], 'text']].sample(10, random_state=0)

## 2. Label check
Ticket counts per class. Classes under `labels.min_class_count` become `ROUTE_TO_HUMAN`.

In [ ]:
df['assignment_group'].value_counts().to_frame('tickets').assign(folded=lambda t: t.tickets < cfg['labels']['min_class_count']['assignment_group'])

## 3. Train, score honestly, save
`promote='no'` saves the model without serving it. Use `'yes'` the first time, `'auto'` afterwards.

In [ ]:
from triage.train import run
result = run(cfg, promote='no')
result['version'], result['comparison']

In [ ]:
from IPython.display import Markdown
Markdown(open(result['report']).read())

## 4. Try your own tickets

In [ ]:
model = registry.load(models_dir, result['version'])
my_tickets = [
    {'number': 'TEST1', 'opened_at': '2026-10-12 09:00', 'short_description': 'Outlook keeps asking for password',
     'description': 'since this morning', 'contact_type': 'Phone', 'impact': '3 - Low'},
]
model.predict(my_tickets).T

## 5. Promote / roll back

In [ ]:
# registry.promote(models_dir, result['version'], reason='approved after review')
# registry.rollback(models_dir)
registry.production_version(models_dir)